In [1]:
from pathlib import Path
import sys
import subprocess
import pandas as pd
import numpy as np
import yaml

current = Path.cwd()
project_root = next(
    (p for p in [current, *current.parents] if (p / "data" / "train.csv").exists()),
    None
)

if project_root is None:
    raise FileNotFoundError("Project root not found. Open this notebook from the project folder.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

data_dir = project_root / "data"

print("Project root:", project_root)

Project root: C:\Users\soura\G137_MP044_MLOps


In [2]:
train = pd.read_csv(data_dir / "train.csv")
later = pd.read_csv(data_dir / "later.csv")
holdout = pd.read_csv(data_dir / "holdout_features.csv")

print("Train shape:", train.shape)
print("Later shape:", later.shape)
print("Holdout shape:", holdout.shape)
print("\nTrain columns:")
print(train.columns.tolist())
print("\nTrain preview:")
display(train.head())

Train shape: (1032, 12)
Later shape: (300, 12)
Holdout shape: (250, 10)

Train columns:
['record_id', 'date', 'cows', 'buffaloes', 'fodder_price_rs', 'temp_c', 'vet_visits', 'feed_kg', 'breed_mix', 'village_cluster', 'litres_paid_in_passbook', 'litres_per_day']

Train preview:


,record_id,date,cows,buffaloes,fodder_price_rs,temp_c,vet_visits,feed_kg,breed_mix,village_cluster,litres_paid_in_passbook,litres_per_day
0,R00001,2026-01-01,127,56,10.4,21.0,9,2329,desi,Cluster B,706.5,697.0
1,R00002,2026-01-01,47,126,33.4,16.3,4,1905,crossbred,Cluster C,667.6,668.4
2,R00003,2026-01-01,31,14,29.0,41.0,7,648,crossbred,Cluster C,276.4,276.0
3,R00004,2026-01-01,129,126,38.1,10.7,3,1878,desi,Cluster A,700.6,694.3
4,R00005,2026-01-01,94,142,12.5,35.6,4,2776,mixed,Cluster C,826.1,822.9


In [3]:
print("MISSING VALUES IN TRAIN")
display(train.isnull().sum().to_frame("missing_count"))

print("DUPLICATE ROWS:", train.duplicated().sum())
print("SENTINEL VALUES (-999):")
for column in train.select_dtypes(include=np.number).columns:
    count = int((train[column] == -999).sum())
    if count:
        print(column, count)

MISSING VALUES IN TRAIN


,missing_count
record_id,0
date,0
cows,0
buffaloes,0
fodder_price_rs,0
temp_c,0
vet_visits,0
feed_kg,0
breed_mix,0
village_cluster,0


DUPLICATE ROWS: 32
SENTINEL VALUES (-999):
fodder_price_rs 35


In [4]:
from src.mp044.data_audit import (
    load_datasets,
    clean_training_data,
    audit_data_issues
)

train, later, holdout = load_datasets(str(data_dir))
findings = audit_data_issues(train, later, holdout)

print("AUTOMATED DATA AUDIT")
for issue, details in findings.items():
    print("\n", issue)
    for key, value in details.items():
        print(f"{key}: {value}")

AUTOMATED DATA AUDIT

 leaking_column
present: True
column: litres_paid_in_passbook

 unit_mixup
present: False
column: None
group_value: None
factor: None

 sentinel_value
present: True
column: fodder_price_rs
value: -999
count: 35

 duplicate_rows
present: True
count: 32

 stuck_sensor
present: True
column: cows
start_date: 2026-02-23
end_date: 2026-03-18

 target_typos
present: False
count: None

 spelling_variants
present: False
column: None
true_levels: None


In [5]:
print("CHECK 1 - LEAKING COLUMN")
print(findings["leaking_column"])

print("\nCHECK 2 - UNIT MIX-UP")
print(findings["unit_mixup"])

print("\nCHECK 3 - SENTINEL VALUE")
print(findings["sentinel_value"])

print("\nCHECK 4 - DUPLICATE ROWS")
print(findings["duplicate_rows"])

print("\nCHECK 5 - STUCK SENSOR")
print(findings["stuck_sensor"])

print("\nCHECK 6 - TARGET TYPOS")
print(findings["target_typos"])

print("\nCHECK 7 - SPELLING VARIANTS")
print(findings["spelling_variants"])

CHECK 1 - LEAKING COLUMN
{'present': True, 'column': 'litres_paid_in_passbook'}

CHECK 2 - UNIT MIX-UP
{'present': False, 'column': None, 'group_value': None, 'factor': None}

CHECK 3 - SENTINEL VALUE
{'present': True, 'column': 'fodder_price_rs', 'value': -999, 'count': 35}

CHECK 4 - DUPLICATE ROWS
{'present': True, 'count': 32}

CHECK 5 - STUCK SENSOR
{'present': True, 'column': 'cows', 'start_date': '2026-02-23', 'end_date': '2026-03-18'}

CHECK 6 - TARGET TYPOS
{'present': False, 'count': None}

CHECK 7 - SPELLING VARIANTS
{'present': False, 'column': None, 'true_levels': None}


In [6]:
train_clean = clean_training_data(train)

print("ORIGINAL TRAIN SHAPE:", train.shape)
print("CLEANED TRAIN SHAPE:", train_clean.shape)
print("DUPLICATES BEFORE:", train.duplicated().sum())
print("DUPLICATES AFTER:", train_clean.duplicated().sum())
print("SENTINELS BEFORE:", int((train["fodder_price_rs"] == -999).sum()))
print("MISSING FODDER PRICES AFTER:",
      int(train_clean["fodder_price_rs"].isna().sum()))
print("ORIGINAL DATA STILL HAS SENTINELS:",
      bool((train["fodder_price_rs"] == -999).any()))

ORIGINAL TRAIN SHAPE: (1032, 12)
CLEANED TRAIN SHAPE: (1000, 12)
DUPLICATES BEFORE: 32
DUPLICATES AFTER: 0
SENTINELS BEFORE: 35
MISSING FODDER PRICES AFTER: 34
ORIGINAL DATA STILL HAS SENTINELS: True


In [7]:
findings_path = project_root / "findings.yaml"

with findings_path.open("r", encoding="utf-8") as file:
    yaml_data = yaml.safe_load(file)

yaml_findings = yaml_data["data_issues"]

all_match = True

print("DATA AUDIT CONSISTENCY CHECK")
print("-" * 45)

for issue, detected in findings.items():
    recorded = yaml_findings[issue]
    if detected == recorded:
        print(issue + ": MATCH")
    else:
        print(issue + ": MISMATCH")
        print("Detected:", detected)
        print("YAML:", recorded)
        all_match = False

print("-" * 45)
print("SUCCESS: All seven checks match findings.yaml." if all_match
      else "ERROR: Some findings do not match findings.yaml.")

DATA AUDIT CONSISTENCY CHECK
---------------------------------------------
leaking_column: MATCH
unit_mixup: MATCH
sentinel_value: MATCH
duplicate_rows: MATCH
stuck_sensor: MATCH
target_typos: MATCH
spelling_variants: MATCH
---------------------------------------------
SUCCESS: All seven checks match findings.yaml.


In [8]:
dvc_status = subprocess.run(
    [sys.executable, "-m", "dvc", "status"],
    cwd=project_root,
    capture_output=True,
    text=True
)

print("DVC STATUS:")
print(dvc_status.stdout.strip() or dvc_status.stderr.strip())

print("\nDVC TRACKING FILES")
for filename in ["data/train.csv.dvc", "data/later.csv.dvc"]:
    file_path = project_root / filename
    print(filename, "FOUND" if file_path.exists() else "MISSING")

DVC STATUS:
Data and pipelines are up to date.

DVC TRACKING FILES
data/train.csv.dvc FOUND
data/later.csv.dvc FOUND


In [9]:
dvc_remotes = subprocess.run(
    [sys.executable, "-m", "dvc", "remote", "list"],
    cwd=project_root,
    capture_output=True,
    text=True
)

print("DVC REMOTES:")
print(dvc_remotes.stdout.strip() if dvc_remotes.stdout.strip()
      else "No DVC remote configured.")

DVC REMOTES:
No DVC remote configured.


In [10]:
print("STAGE 1 DATA LEAD SUMMARY")
print("Dataset loading: completed")
print("Required data checks:", len(findings))
print("Findings YAML consistency:", "PASS" if all_match else "FAIL")
print("Original training data unchanged:", 
      bool((train["fodder_price_rs"] == -999).any()))
print("DVC tracking files present:",
      all((project_root / f).exists()
          for f in ["data/train.csv.dvc", "data/later.csv.dvc"]))

STAGE 1 DATA LEAD SUMMARY
Dataset loading: completed
Required data checks: 7
Findings YAML consistency: PASS
Original training data unchanged: True
DVC tracking files present: True


In [13]:
import subprocess

result = subprocess.run(
    ["git", "commit", "-m", "Complete Stage 1 data audit and DVC tracking"],
    cwd=project_root,
    capture_output=True,
    text=True
)

print(result.stdout)
print(result.stderr)

if result.returncode == 0:
    print("COMMIT SUCCESSFUL")
else:
    print("COMMIT FAILED")

[main 028efde] Complete Stage 1 data audit and DVC tracking
 10 files changed, 14051 insertions(+)
 create mode 100644 .dvc/.gitignore
 create mode 100644 .dvc/config
 create mode 100644 .dvcignore
 create mode 100644 .gitignore
 create mode 100644 data/.gitignore
 create mode 100644 data/later.csv.dvc
 create mode 100644 data/train.csv.dvc
 create mode 100644 findings.yaml
 create mode 100644 notebooks/stage1_data_audit.ipynb
 create mode 100644 src/mp044/data_audit.py


COMMIT SUCCESSFUL


In [14]:
result = subprocess.run(
    ["git", "status", "--short"],
    cwd=project_root,
    capture_output=True,
    text=True
)

print("GIT STATUS:")
print(result.stdout if result.stdout.strip() else "Working tree clean.")

GIT STATUS:
 M notebooks/stage1_data_audit.ipynb
?? GROUP.md
?? PROBLEM.md
?? STUDENT_HANDBOOK.md
?? data/holdout_features.csv
?? notebooks/stage1_data_audit_code_only.ipynb

